# Week 4 – Text Embeddings
## Generating and Comparing Text Embeddings using Sentence Transformers

**Objective:** Generate sentence embeddings and compare sentence similarity using cosine similarity.

**Model:** `all-MiniLM-L6-v2`


## 1. Import the libraries

If a package is missing, run `python -m pip install -r requirements.txt` in the project terminal, then restart the notebook kernel.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


## 2. Load the dataset

In [ ]:
BASE_DIR = Path.cwd()
# If the notebook is opened from the notebook folder, use its parent as the project folder.
if not (BASE_DIR / "dataset").exists() and (BASE_DIR.parent / "dataset").exists():
    BASE_DIR = BASE_DIR.parent

DATASET_DIR = BASE_DIR / "dataset"
df = pd.read_csv(DATASET_DIR / "generated_text.csv")
df.head()


## 3. Generate sentence embeddings

In [ ]:
sentences = df["text"].dropna().astype(str).tolist()
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    sentences,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Number of sentences:", len(sentences))
print("Embedding shape:", embeddings.shape)
print("Embedding dimension:", embeddings.shape[1])
np.save(DATASET_DIR / "embeddings.npy", embeddings)


## 4. Calculate cosine similarity

In [ ]:
# Normalized embeddings allow cosine similarity to be calculated using a dot product.
similarity_matrix = embeddings @ embeddings.T
similarity_matrix_df = pd.DataFrame(
    similarity_matrix,
    index=df["sentence_id"],
    columns=df["sentence_id"]
)
similarity_matrix_df.head()


## 5. Save all unique sentence-pair scores and display the top five

In [ ]:
rows = []
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        rows.append({
            "sentence_id_1": int(df.iloc[i]["sentence_id"]),
            "sentence_id_2": int(df.iloc[j]["sentence_id"]),
            "sentence_1": sentences[i],
            "sentence_2": sentences[j],
            "cosine_similarity": float(similarity_matrix[i, j])
        })

scores = pd.DataFrame(rows).sort_values("cosine_similarity", ascending=False)
scores.to_csv(DATASET_DIR / "similarity_scores.csv", index=False)
scores.head(5)


## 6. Compare selected sentence pairs

In [ ]:
pairs = pd.read_csv(DATASET_DIR / "sentence_pairs.csv")
id_to_position = {int(row["sentence_id"]): pos for pos, (_, row) in enumerate(df.iterrows())}

pair_results = []
for _, pair in pairs.iterrows():
    id1, id2 = int(pair["sentence_id_1"]), int(pair["sentence_id_2"])
    i, j = id_to_position[id1], id_to_position[id2]
    pair_results.append({
        "sentence_id_1": id1,
        "sentence_id_2": id2,
        "sentence_1": sentences[i],
        "sentence_2": sentences[j],
        "cosine_similarity": float(similarity_matrix[i, j])
    })

pair_results_df = pd.DataFrame(pair_results).sort_values(
    "cosine_similarity", ascending=False
)
pair_results_df.to_csv(DATASET_DIR / "sentence_pairs.csv", index=False)
pair_results_df


## 7. Observation and conclusion

Text embeddings represent sentence meaning as numerical vectors. The model used here produces 384-dimensional vectors. Cosine similarity helps compare the semantic relationship between sentence pairs; higher values generally indicate greater similarity. Embeddings are useful for semantic search and Retrieval-Augmented Generation (RAG).
